# The Visual ROI — Analysis Notebook

Università degli Studi di Milano · Data Science for Economics

This notebook reproduces every number, figure and table in the thesis *The Visual ROI: Quantifying the Impact of AI-Derived Visual Sentiment on Social Media Engagement and Revenue Performance* (Honest Albert Temu, MSc Data Science for Economics, 2025-2026). It runs from local files (Jupyter, VS Code or Google Colab).

**Data.** The daily sales log and the Instagram media archive belong to Cold Culture and are not part of this repository. `master_dataset.csv` (79 rows: date, revenue, ad spend, weekend and drop flags, visual score) is available from the author on request, subject to the brand's permission; place it in the repository root to run the analysis.

**Expected folder layout** (open VS Code on the thesis folder):

```
visual-roi-thesis/
├── Thesis.ipynb                 ← this notebook
├── master_dataset.csv           ← frozen analysis dataset (on request; not in the repository)
├── data/
│   ├── cculture.zip             ← Instagram media archive (only needed to re-score)
│   └── coldculture_DATA.csv     ← daily sales, spend, weekend and drop flags (only needed to rebuild)
├── figures/                     ← every thesis figure is written here
│   └── exploratory/             ← working charts not used in the thesis
└── results/                     ← key numbers and checks as CSV / JSON
    └── tables/                  ← every thesis table as CSV 
```

**Run modes** (switches in the next cell):

| Switch | Default | What it does |
|---|---|---|
| `RUN_CLIP_SCORING` | `False` | Re-scores all media with CLIP (slow, needs `torch` + `clip`). |
| `REBUILD_MASTER` | `False` | Rebuilds `master_dataset.csv` from the scores and the sales file. |

With both off, the notebook reads the frozen `master_dataset.csv` and regenerates the figures and numbers exactly as they appear in the thesis.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────
from pathlib import Path
import hashlib, json, warnings
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
if not (ROOT / 'master_dataset.csv').exists() and (ROOT.parent / 'master_dataset.csv').exists():
    ROOT = ROOT.parent          # notebook kept in a sub-folder such as notebooks/

DATA_DIR    = ROOT / 'data'
FIG_DIR     = ROOT / 'figures'
EXPL_DIR    = FIG_DIR / 'exploratory'
RESULTS_DIR = ROOT / 'results'

MEDIA_ZIP       = DATA_DIR / 'cculture.zip'
MEDIA_DIR       = DATA_DIR / 'cculture_processed_media'
SALES_CSV       = DATA_DIR / 'coldculture_DATA.csv'
SCORES_CSV      = DATA_DIR / 'final_sentiment_score.csv'
ITEM_SCORES_CSV = DATA_DIR / 'clip_item_scores.csv'
MASTER_CSV      = ROOT / 'master_dataset.csv'

RUN_CLIP_SCORING      = False
REBUILD_MASTER        = False

# Fingerprint of the dataset behind the thesis
FROZEN_MASTER_SHA256 = 'c62d46cfec5ea9573e2742bab6387771054d38b442a689ddf9906466aec293a4'

for d in (DATA_DIR, FIG_DIR, EXPL_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('Project folder:', ROOT)
# ── Timing of each stage of the pipeline (reported in Table 2 of the thesis) ──
import time
TIMINGS, _t0 = {}, {}
def tic(stage):
    _t0[stage] = time.perf_counter()
def toc(stage):
    TIMINGS[stage] = round(time.perf_counter() - _t0.pop(stage), 2)
    print(f'[time] {stage}: {TIMINGS[stage]} s')


## Setup: imports, style and helpers

In [ ]:
# ══════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.dates as mdates
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Liberation Serif', 'DejaVu Serif'],   # matches the thesis text
    'mathtext.fontset': 'stix',
    'font.size': 9.5,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.20,
    'grid.linestyle': '--',
    'figure.dpi': 180,
    'savefig.bbox': 'tight',
    'savefig.facecolor': 'white',
})

TEAL  = '#006B6B'
SLATE = '#3A4A5C'
RED   = '#C0392B'
GOLD  = '#B8860B'
GREY  = '#AAAAAA'
GREEN = '#1A7A4A'
LIGHT = '#F0F5F8'
MID   = '#D0E4EE'

DROP_DATES = pd.to_datetime(['2025-10-24', '2025-11-28', '2025-12-12'])

# ── Helpers ────────────────────────────────────────────────────────────
def euro(x, pos=None):
    return f'€{x:,.0f}'

def sig_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

def fmt_p(p):
    return '< 0.001' if p < 0.001 else f'{p:.3f}'

def table_header(ax, col_x, col_w, cols, color=SLATE):
    ax.add_patch(FancyBboxPatch(
        (0, 0.85), 1, 0.15, transform=ax.transAxes,
        boxstyle='square,pad=0', facecolor=color))
    for x, w, c in zip(col_x, col_w, cols):
        ax.text(x + w/2, 0.92, c, ha='center', va='center',
                fontsize=8.5, fontweight='bold', color='white',
                transform=ax.transAxes)

def table_row(ax, y0, row_h, bg):
    ax.add_patch(FancyBboxPatch(
        (0, y0 - 0.01), 1, row_h, transform=ax.transAxes,
        boxstyle='square,pad=0', facecolor=bg, zorder=0))

def save(name, folder=None):
    out = (folder or FIG_DIR) / name
    plt.savefig(out)
    plt.show()
    plt.close()
    print(f'  ✓  {out.relative_to(ROOT)}')

TABLE_DIR = RESULTS_DIR / 'tables'
TABLE_DIR.mkdir(parents=True, exist_ok=True)

## Stage 1 — CLIP visual scoring *(optional, off by default)*

Scores every image and sampled video frame against two prompts and keeps the probability of the product-focused prompt. Settings are identical to the original run, so a re-run reproduces the same method. Two things can make re-computed scores differ slightly from the published ones:

- The original run used a Google Colab CPU runtime, where CLIP runs in 32-bit floating point. On a CUDA GPU, OpenAI's CLIP package loads the model in half precision, so probabilities can differ in the third or fourth decimal.
- Different OpenCV versions can decode video frames slightly differently.

Item-level scores are saved to `data/clip_item_scores.csv` so that any difference can be traced to individual files. The time taken by model loading and by scoring is recorded for Table 2 of the thesis.


In [ ]:
# ── CLIP scoring functions (imports are inside so the analysis runs without torch) ──
TEXT_PROMPTS = [
    "A clear fashion photography shot where the streetwear clothing and outfit are the obvious central focus.",
    "An atmospheric lifestyle, scenery, or mood photo where the clothing is obscured, hidden, or not the main subject.",
]
FRAME_STEP = 60          # every 60th frame = one frame every 2 s at 30 fps (as in the Colab run)
IMAGE_EXT = ('.jpg', '.jpeg', '.png', '.webp')
VIDEO_EXT = ('.mp4', '.mov')
_clip = {}

def load_clip():
    if not _clip:
        import torch, clip
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        model, preprocess = clip.load('ViT-B/32', device=device)
        _clip.update(torch=torch, model=model, preprocess=preprocess, device=device,
                     tokens=clip.tokenize(TEXT_PROMPTS).to(device))
        print('CLIP ViT-B/32 loaded on', device)
    return _clip

def clip_score(pil_img):
    c = load_clip()
    x = c['preprocess'](pil_img).unsqueeze(0).to(c['device'])
    with c['torch'].no_grad():
        logits, _ = c['model'](x, c['tokens'])
        return float(logits.softmax(dim=-1).cpu().numpy()[0][0])

def score_video(path, step=FRAME_STEP):
    import cv2
    from PIL import Image
    cap, scores, i = cv2.VideoCapture(str(path)), [], 0
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        if i % step == 0:
            scores.append(clip_score(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))))
        i += 1
    cap.release()
    return (float(np.mean(scores)), len(scores)) if scores else (None, 0)

def media_files():
    if not MEDIA_DIR.exists():
        import zipfile
        print(f'Extracting {MEDIA_ZIP.name} ...')
        with zipfile.ZipFile(MEDIA_ZIP) as z:
            z.extractall(MEDIA_DIR)
    return sorted(p for p in MEDIA_DIR.rglob('*')
                  if p.is_file() and not p.name.startswith('._') and '__MACOSX' not in p.parts
                  and p.suffix.lower() in IMAGE_EXT + VIDEO_EXT)

def date_from_name(name):
    # files are named ..._YYYY-MM-DD.ext ; the date is the part after the last underscore
    return pd.to_datetime(name.split('_')[-1].split('.')[0], errors='coerce')

In [ ]:
# ── Run the scoring ───────────────────────────────────────────────────
if RUN_CLIP_SCORING:
    from PIL import Image
    tic('Model loading'); load_clip(); toc('Model loading')
    files = media_files()
    tic('CLIP scoring')
    rows = []
    for p in files:
        try:
            if p.suffix.lower() in IMAGE_EXT:
                rows.append(dict(File=p.name, Type='Image', Score=clip_score(Image.open(p).convert('RGB')), Frames=1))
            else:
                s, k = score_video(p)
                if s is not None:
                    rows.append(dict(File=p.name, Type='Video', Score=s, Frames=k))
        except Exception as e:
            print(f'Error processing {p.name}: {e}')
    toc('CLIP scoring')
    tic('Daily aggregation and merge')
    items = pd.DataFrame(rows)
    items['Date'] = items['File'].apply(date_from_name)
    items.to_csv(ITEM_SCORES_CSV, index=False)
    daily = (items.groupby('Date')['Score'].mean().round(4)
                  .rename('Visual_Score').reset_index())
    daily.to_csv(SCORES_CSV, index=False)
    toc('Daily aggregation and merge')
    print(f'{len(items)} media items scored on {daily.shape[0]} posting days -> {SCORES_CSV.relative_to(ROOT)}')
else:
    print('Stage 1 skipped (RUN_CLIP_SCORING = False).')

## Stage 2 — Master dataset

`master_dataset.csv` is the frozen dataset behind every number in the thesis. It is only rebuilt when `REBUILD_MASTER = True`: daily scores are merged onto the sales log, forward-filled across non-posting days, and back-filled for the first day(s) before the first post. The fingerprint check below warns if the file no longer matches the one the thesis was written from.

In [ ]:
if REBUILD_MASTER:
    scores = pd.read_csv(SCORES_CSV, parse_dates=['Date'])[['Date', 'Visual_Score']]
    sales  = pd.read_csv(SALES_CSV,  parse_dates=['Date'])
    master = (sales.merge(scores, on='Date', how='left')
                   .sort_values('Date').reset_index(drop=True))
    master['Visual_Score'] = master['Visual_Score'].ffill().bfill()
    master.to_csv(MASTER_CSV, index=False)
    print('master_dataset.csv rebuilt; missing scores:', int(master['Visual_Score'].isna().sum()))

sha = hashlib.sha256(MASTER_CSV.read_bytes()).hexdigest()
if sha == FROZEN_MASTER_SHA256:
    print('✓ master_dataset.csv matches the dataset used for the thesis.')
else:
    print('⚠ master_dataset.csv has CHANGED since the thesis was written.\n'
          '  Numbers in the thesis may need updating; see the key-number check at the end.')

## Stage 3 — Models (Chapters 3–5)

Ordinary least squares with OLS and HC3 standard errors for Models 1–4, plus the diagnostics quoted in the text.

In [ ]:
tic('Regressions and diagnostics')
# ══════════════════════════════════════════════════════════════════════
df79 = pd.read_csv(MASTER_CSV)
df79['Date'] = pd.to_datetime(df79['Date'])
df79 = df79.sort_values('Date').reset_index(drop=True)

# Mark posting vs forward-fill days
df79['vs_diff']   = df79['Visual_Score'].diff()
df79['is_new']    = (df79['vs_diff'] != 0) | (df79.index == 0)

# Working dataset (lag drops first row)
df = df79.copy()
df['Lag1'] = df['Visual_Score'].shift(1)
df = df.dropna(subset=['Lag1']).reset_index(drop=True)

y = df['Daily_Revenue'].values
n = len(y)

# Adstock (λ = 0.5)
ads = df['Ad_Spend'].values
adstock = np.zeros(n)
adstock[0] = ads[0]
for i in range(1, n):
    adstock[i] = ads[i] + 0.5 * adstock[i-1]


def ols(X, y):
    n, k = X.shape
    b    = np.linalg.lstsq(X, y, rcond=None)[0]
    yhat = X @ b
    e    = y - yhat
    ss_r = float(e @ e)
    ss_t = float(((y - y.mean())**2).sum())
    r2   = 1 - ss_r / ss_t
    r2a  = 1 - (1 - r2) * (n - 1) / (n - k)
    s2   = ss_r / (n - k)
    cov  = s2 * np.linalg.inv(X.T @ X)
    se   = np.sqrt(np.diag(cov))
    t    = b / se
    pv   = 2 * stats.t.sf(np.abs(t), n - k)
    # HC3
    h       = np.diag(X @ np.linalg.inv(X.T @ X) @ X.T)
    e_hc3   = e / (1 - h)
    meat    = (X * (e_hc3**2)[:, None]).T @ X
    bread   = np.linalg.inv(X.T @ X)
    cov_h   = bread @ meat @ bread
    se_h    = np.sqrt(np.diag(cov_h))
    t_h     = b / se_h
    pv_h    = 2 * stats.t.sf(np.abs(t_h), n - k)
    return dict(b=b, se=se, t=t, pv=pv, r2=r2, r2a=r2a,
                se_h=se_h, t_h=t_h, pv_h=pv_h,
                aic=n*np.log(ss_r/n)+2*k,
                bic=n*np.log(ss_r/n)+k*np.log(n),
                rmse=np.sqrt(ss_r/n),
                dw=float(np.sum(np.diff(e)**2)/np.sum(e**2)),
                e=e, yhat=yhat, k=k, ss_r=ss_r)


X1 = np.column_stack([np.ones(n), df['Ad_Spend'],  df['Is_Weekend'], df['Is_Drop']])
X2 = np.column_stack([np.ones(n), df['Ad_Spend'],  df['Is_Weekend'], df['Is_Drop'], df['Visual_Score']])
X3 = np.column_stack([np.ones(n), df['Ad_Spend'],  df['Is_Weekend'], df['Is_Drop'], df['Lag1']])
X4 = np.column_stack([np.ones(n), adstock,         df['Is_Weekend'], df['Is_Drop'], df['Lag1']])

m1, m2, m3, m4 = ols(X1,y), ols(X2,y), ols(X3,y), ols(X4,y)
t_crit = stats.t.ppf(0.975, n - 5)

print('Models fitted.')


# ── Extra statistics used in captions and the key-number check ────────
def vif(X, j):
    xj, others = X[:, j], np.delete(X, j, axis=1)
    Z = np.column_stack([np.ones(len(xj)), others])
    resid = xj - Z @ np.linalg.lstsq(Z, xj, rcond=None)[0]
    return 1 / (resid @ resid / ((xj - xj.mean()) ** 2).sum())

def breusch_pagan(X, e):
    e2 = e ** 2
    fit = X @ np.linalg.lstsq(X, e2, rcond=None)[0]
    lm = len(e2) * (1 - ((e2 - fit) ** 2).sum() / ((e2 - e2.mean()) ** 2).sum())
    return lm, 1 - stats.chi2.cdf(lm, X.shape[1] - 1)

F_13   = ((m1['ss_r'] - m3['ss_r']) / 1) / (m3['ss_r'] / (n - 5))
p_F13  = 1 - stats.f.cdf(F_13, 1, n - 5)
bp_stat, bp_p = breusch_pagan(X3, m3['e'])
sw_W, sw_p = stats.shapiro(m3['e'])
VIF_M3 = {nm: vif(X3[:, 1:], j) for j, nm in enumerate(['Ad_Spend', 'Is_Weekend', 'Is_Drop', 'Visual_Lag1'])}
VIF_M4_vis = vif(X4[:, 1:], 3)

_nd = df[df['Is_Drop'] == 0]
_Xc = np.column_stack([np.ones(len(_nd)), _nd['Is_Weekend']])
_res = lambda v: v - _Xc @ np.linalg.lstsq(_Xc, v, rcond=None)[0]
r_partial_nd = stats.pearsonr(_res(_nd['Daily_Revenue'].values), _res(_nd['Lag1'].values))[0]

print(f"M3: R² = {m3['r2']:.4f} | β(Visual_Lag1) = €{m3['b'][4]:,.2f} | F(1,{n-5}) = {F_13:.2f} | "
      f"BP = {bp_stat:.3f} (p = {bp_p:.4f}) | VIF(Visual_Lag1) = {VIF_M3['Visual_Lag1']:.2f}")

In [ ]:
# ── Optional cross-check with statsmodels (same estimates, standard regression table) ──
try:
    import statsmodels.api as sm
    _X = sm.add_constant(df[['Ad_Spend', 'Is_Weekend', 'Is_Drop', 'Lag1']].rename(columns={'Lag1': 'Visual_Lag1'}))
    print(sm.OLS(df['Daily_Revenue'], _X).fit(cov_type='HC3').summary())
except ImportError:
    print('statsmodels not installed; skipping the cross-check.')
toc('Regressions and diagnostics')


## Chapter 1 figure

Figure 1.1, conceptual framework (diagram, no data).

In [ ]:
tic('Figures and tables')
from matplotlib.patches import Rectangle, FancyArrowPatch

BLACK, DGREY = '#000000', '#555555'
fig, ax = plt.subplots(figsize=(12.5, 5.8))
ax.set_xlim(0, 12.5); ax.set_ylim(0, 5.8); ax.axis('off')

def box(x, y, w, h, title, sub, dashed=False):
    col = DGREY if dashed else BLACK
    ax.add_patch(Rectangle((x, y), w, h, lw=1.2, ec=col, fc='white', ls='--' if dashed else '-'))
    ax.text(x + w/2, y + h*0.68, title, ha='center', va='center', fontsize=10.5, fontweight='bold', color=col)
    ax.text(x + w/2, y + h*0.32, sub, ha='center', va='center', fontsize=9, color=col, linespacing=1.3)

def arrow(x1, y1, x2, y2, dashed=False):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='-|>', mutation_scale=14, lw=1.2,
                                 color=DGREY if dashed else BLACK, ls='--' if dashed else '-'))

W, H, Y = 2.6, 1.3, 2.6
xs = [0.1, 3.3, 6.5, 9.7]
box(xs[0], Y, W, H, 'Brand visual content', 'Instagram images,\ncarousels and Reels')
box(xs[1], Y, W, H, 'CLIP ViT-B/32', 'Zero-shot: product-focused\nvs. atmospheric prompt')
box(xs[2], Y, W, H, 'Daily visual score', 'Averaged, forward-filled,\nlagged one day (t−1)')
box(xs[3], Y, W, H, 'Daily revenue (t)', 'Direct-to-consumer\nweb store sales')
mid = Y + H/2
for a, b in [(0, 1), (1, 2), (2, 3)]:
    arrow(xs[a] + W, mid, xs[b], mid)
ax.text((xs[2] + W + xs[3]) / 2, Y - 0.25, 'Visual ROI (β₄)', ha='center', fontsize=9, style='italic')

box(6.2, 4.45, 3.2, 0.95, 'Engagement and attention', 'theorised mechanism (not observed)', dashed=True)
arrow(7.8, Y + H, 7.8, 4.45, dashed=True)
arrow(9.4, 4.92, 11.0, Y + H, dashed=True)
ax.text(10.35, 4.75, 'processing fluency', ha='left', fontsize=8.6, style='italic', color=DGREY)

box(9.7, 0.3, 2.6, 1.5, 'Controls', 'Ad spend (β₁)\nWeekend (β₂)\nProduct drop (β₃)')
arrow(11.0, 1.8, 11.0, Y)

ax.text(0.3, 1.55, 'Measurement layer', fontsize=9.5, fontweight='bold')
ax.text(0.3, 1.18, 'Chapter 3: from raw content to a daily score', fontsize=9)
ax.text(0.3, 0.72, 'Estimation layer', fontsize=9.5, fontweight='bold')
ax.text(0.3, 0.35, 'Chapters 4–5: OLS marketing mix model, N = 78 days', fontsize=9)
save('fig_ch1_fig11_conceptual_framework.png')

## Chapter 2 figure

Figure 2, the CLIP architecture and its zero-shot use in this thesis (diagram, no data; drawn after Radford et al., 2021).

In [ ]:
# Figure 2: CLIP architecture (diagram, no data) (contrastive pre-training) and its zero-shot use in this thesis.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch

_rc = plt.rc_context({'font.family': 'serif',
                      'font.serif': ['Times New Roman', 'Liberation Serif', 'DejaVu Serif'],
                      'mathtext.fontset': 'stix', 'figure.dpi': 200})
_rc.__enter__()
BLACK, GREY, LIGHT = '#000000', '#555555', '#d9d9d9'

fig, ax = plt.subplots(figsize=(12.5, 6.4))
ax.set_xlim(0, 12.5); ax.set_ylim(0, 6.4); ax.axis('off')

def box(x, y, w, h, text, fs=9.5, bold=False, fill='white', dashed=False, col=BLACK):
    ax.add_patch(Rectangle((x, y), w, h, lw=1.1, ec=col, fc=fill, ls='--' if dashed else '-'))
    ax.text(x + w / 2, y + h / 2, text, ha='center', va='center', fontsize=fs,
            fontweight='bold' if bold else 'normal', color=col, linespacing=1.25)

def arrow(x1, y1, x2, y2, col=BLACK):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='-|>', mutation_scale=11, lw=1.0, color=col))

# ── (a) contrastive pre-training ────────────────────────────────────────
ax.text(0.1, 6.15, '(a) Contrastive pre-training on 400 million image–text pairs', fontsize=11, fontweight='bold')
box(0.1, 4.75, 1.6, 0.75, 'caption\n“a photo of …”', fs=9)
box(2.0, 4.75, 1.6, 0.75, 'Text encoder\n(Transformer)', bold=True)
arrow(1.7, 5.125, 2.0, 5.125)
box(0.1, 2.35, 1.6, 0.75, 'image', fs=9)
box(2.0, 2.35, 1.6, 0.75, 'Image encoder\n(ViT-B/32)', bold=True)
arrow(1.7, 2.725, 2.0, 2.725)

# text embeddings along the top of the matrix, image embeddings down the side
N, cell, x0, y0 = 5, 0.46, 4.35, 0.75
for j in range(N):
    box(x0 + j * cell, y0 + N * cell + 0.15, cell, 0.4, f'$T_{j+1}$', fs=9, fill=LIGHT)
for i in range(N):
    box(x0 - 0.55, y0 + (N - 1 - i) * cell, 0.4, cell, f'$I_{i+1}$', fs=9, fill=LIGHT)
    for j in range(N):
        diag = i == j
        box(x0 + j * cell, y0 + (N - 1 - i) * cell, cell, cell,
            f'$I_{i+1}\\!\\cdot\\!T_{j+1}$' if diag else '', fs=6.5, fill='#808080' if diag else 'white',
            col=BLACK)
arrow(3.6, 5.125, x0 + N * cell / 2, y0 + N * cell + 0.57)
arrow(3.6, 2.725, x0 - 0.57, 2.725)
ax.text(x0 + N * cell / 2, 0.35, 'matching pairs (diagonal) pulled together,\nall other pairs pushed apart',
        ha='center', va='center', fontsize=8.8, style='italic')

ax.plot([7.15, 7.15], [0.2, 6.2], color=GREY, lw=0.8, ls=':')

# ── (b) zero-shot scoring used in this thesis ───────────────────────────
ax.text(7.35, 6.15, '(b) Zero-shot scoring in this thesis', fontsize=11, fontweight='bold')
box(7.35, 4.95, 2.35, 0.85, 'Prompt 1: product-focused\nfashion shot', fs=8.8)
box(7.35, 3.95, 2.35, 0.85, 'Prompt 2: atmospheric\nlifestyle / mood photo', fs=8.8)
box(10.0, 4.45, 1.5, 0.85, 'Text encoder', bold=True, fs=9)
arrow(9.7, 5.375, 10.0, 5.0); arrow(9.7, 4.375, 10.0, 4.75)
box(7.35, 2.0, 2.35, 1.2, 'Post image or video frame\n224 × 224 px, split into\n7 × 7 patches of 32 × 32 px', fs=8.6)
box(10.0, 2.17, 1.5, 0.85, 'Image encoder\n(ViT-B/32)', bold=True, fs=9)
arrow(9.7, 2.6, 10.0, 2.6)
box(11.75, 3.25, 0.65, 0.8, 'cosine\nsim.', fs=8.4)
arrow(11.5, 4.87, 11.75, 3.9); arrow(11.5, 2.6, 11.75, 3.4)
box(9.55, 0.55, 2.85, 1.0, 'softmax over the two prompts\n$\\mathrm{Visual\\_Score} = P(\\mathrm{prompt\\ 1}\\,|\\,\\mathrm{image})$', fs=8.8)
arrow(12.07, 3.25, 12.07, 1.55)
ax.text(8.5, 1.05, '512-dimensional\nshared embedding\nspace', ha='center', va='center', fontsize=8.6, style='italic', color=GREY)

out = FIG_DIR / 'fig_ch2_fig21_clip_architecture.png'
plt.savefig(out, bbox_inches='tight', facecolor='white')
plt.close(fig)
_rc.__exit__(None, None, None)
print('saved', out.relative_to(ROOT))


## Figure 3 — brand images by visual score *(needs the media archive)*

Builds the panel of real Cold Culture posts for Figure 3 from `data/cculture.zip`: the lowest-scoring day (9 Dec), the low-score episode (20 Nov), a day near the 25th percentile and one of the highest-scoring days. Leave `PANEL_DATES = None` to pick these automatically, or list the dates you want. Skipped when the archive is not in `data/`. Cold Culture has consented to the reproduction of its images in the thesis.

In [ ]:
# Archive layout (as in the Colab run): media files directly in cculture.zip, each named
# <anything>_YYYY-MM-DD.<ext> with ext in .jpg .jpeg .png .webp .mp4 .mov; the date is read after the last '_'.
PANEL_DATES = None          # e.g. ['2025-12-09', '2025-11-20', '2025-10-27', '2025-12-08']

def thumbnail(path):
    from PIL import Image
    if path.suffix.lower() in IMAGE_EXT:
        return Image.open(path).convert('RGB')
    import cv2                                   # video: use the middle frame
    cap = cv2.VideoCapture(str(path)); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, max(n // 2, 0)); ok, frame = cap.read(); cap.release()
    return Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)) if ok else None

if MEDIA_ZIP.exists() or MEDIA_DIR.exists():
    files_by_day = {}
    for f in media_files():
        files_by_day.setdefault(date_from_name(f.name), []).append(f)
    posting = df79[df79['is_new']].set_index('Date')['Visual_Score']
    posting = posting[[d in files_by_day for d in posting.index]]
    if PANEL_DATES:
        days = [pd.Timestamp(d) for d in PANEL_DATES]
    else:                                        # lowest, low episode, ~25th percentile, highest
        days = []
        for target in [posting.min(), 0.231, df['Lag1'].quantile(0.25), posting.max()]:
            d = (posting.drop(days, errors='ignore') - target).abs().idxmin()
            days.append(d)
    days = sorted(days, key=lambda d: posting.get(d, df79.set_index('Date')['Visual_Score'][d]))
    items = pd.read_csv(ITEM_SCORES_CSV, parse_dates=['Date']) if ITEM_SCORES_CSV.exists() else None

    rows = 2 if len(days) == 4 else 1               # four posts: a 2 x 2 grid, readable at page width
    cols = len(days) // rows
    fig, axes = plt.subplots(rows, cols, figsize=(3.6 * cols, 4.0 * rows))
    for ax, d in zip(np.ravel(axes), days):
        day_score = df79.set_index('Date')['Visual_Score'][d]
        # default: the cover (slide 1) of the day's post, i.e. what followers saw first
        slide = lambda f: int(f.stem.split('_')[-2]) if len(f.stem.split('_')) >= 5 and f.stem.split('_')[-2].isdigit() else 1
        cands = sorted(files_by_day[d], key=lambda f: (slide(f), f.name))
        if items is not None:                    # the post whose own score is closest to the day's score
            sc = items[items['Date'] == d].set_index('File')['Score']
            cands = sorted(cands, key=lambda f: abs(sc.get(f.name, 9) - day_score))
        img = next((im for im in (thumbnail(f) for f in cands) if im is not None), None)
        if img is not None:
            w, h = img.size; s = min(w, h)       # centre square crop
            ax.imshow(img.crop(((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2)))
        ax.set_title(f"{d.strftime('%d %b %Y')}\nVisual_Score = {day_score:.3f}", fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    save('fig_ch2_fig22_visual_quality_spectrum.png')
else:
    print('Figure 3 skipped: put cculture.zip in data/ to build the brand image panel.')

## Chapter 4 figures

In [ ]:
# ══════════════════════════════════════════════════════════════════════
print('\n── Chapter 4 ──')

# ── Figure 4.1: Revenue Distribution ──────────────────────────────────
# §4.2.1 — shows the two-regime structure (drop vs non-drop).

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Daily Revenue Distribution', fontweight='bold', fontsize=11)

# Left: full distribution
ax = axes[0]
ax.hist(df79['Daily_Revenue'], bins=22, color=TEAL, edgecolor='white', alpha=0.85)
ax.axvline(df79['Daily_Revenue'].mean(),   color=RED,  lw=1.8, linestyle='--', label=f'Mean €{df79["Daily_Revenue"].mean():,.0f}')
ax.axvline(df79['Daily_Revenue'].median(), color=GOLD, lw=1.8, linestyle=':',  label=f'Median €{df79["Daily_Revenue"].median():,.0f}')
ax.set_xlabel('Daily Revenue (€)', fontsize=10)
ax.set_ylabel('Frequency', fontsize=10)
ax.set_title('All 79 Days', fontsize=10)
ax.xaxis.set_major_formatter(plt.FuncFormatter(euro))
ax.legend(fontsize=9)

# Right: non-drop only
nd_rev = df79[df79['Is_Drop']==0]['Daily_Revenue']
ax = axes[1]
ax.hist(nd_rev, bins=18, color=SLATE, edgecolor='white', alpha=0.85)
ax.axvline(nd_rev.mean(),   color=RED,  lw=1.8, linestyle='--', label=f'Mean €{nd_rev.mean():,.0f}')
ax.axvline(nd_rev.median(), color=GOLD, lw=1.8, linestyle=':',  label=f'Median €{nd_rev.median():,.0f}')

sw, swp = stats.shapiro(nd_rev)
ax.text(0.97, 0.95, f'Shapiro–Wilk W = {sw:.3f}\np = {swp:.3f}  (normal)',
        transform=ax.transAxes, ha='right', va='top', fontsize=8.5,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85, edgecolor=MID))
ax.set_xlabel('Daily Revenue (€)', fontsize=10)
ax.set_title('Non-Drop Days Only (N = 76)', fontsize=10)
ax.xaxis.set_major_formatter(plt.FuncFormatter(euro))
ax.legend(fontsize=9)

plt.tight_layout()
save('fig_ch4_fig41_revenue_distribution.png')


# ── Figure 4.2: Visual_Lag1 vs Revenue (non-drop) ─────────────────────
# §4.3.3 — partial r = 0.877 after controlling for weekend.

fig, ax = plt.subplots(figsize=(8, 5.5))
nd   = df[df['Is_Drop'] == 0].copy()
wkd  = nd[nd['Is_Weekend'] == 0]
wke  = nd[nd['Is_Weekend'] == 1]

ax.scatter(wkd['Lag1'], wkd['Daily_Revenue'],
           color=SLATE, alpha=0.75, s=52, zorder=3, label='Weekday')
ax.scatter(wke['Lag1'], wke['Daily_Revenue'],
           color=TEAL, alpha=0.85, s=62, marker='D', zorder=3, label='Weekend')

x_fit = np.linspace(nd['Lag1'].min()-0.02, nd['Lag1'].max()+0.02, 200)
Xf    = np.column_stack([np.ones(len(nd)), nd['Lag1']])
bf    = np.linalg.lstsq(Xf, nd['Daily_Revenue'].values, rcond=None)[0]
ax.plot(x_fit, bf[0] + bf[1]*x_fit, color=RED, lw=2, zorder=4,
        label='OLS trend (non-drop)')

se_fit = []
for xi in x_fit:
    xv = np.array([1, xi])
    se_fit.append(np.sqrt(
        xv @ np.linalg.inv(Xf.T @ Xf) @ xv *
        np.sum((nd['Daily_Revenue'].values - Xf@bf)**2) / (len(nd)-2)))
se_fit = np.array(se_fit)
ax.fill_between(x_fit,
                bf[0]+bf[1]*x_fit - 1.96*se_fit,
                bf[0]+bf[1]*x_fit + 1.96*se_fit,
                color=RED, alpha=0.10)

r_raw, _ = stats.pearsonr(nd['Lag1'], nd['Daily_Revenue'])
# Partial r controlling for Is_Weekend
Xc  = np.column_stack([np.ones(len(nd)), nd['Is_Weekend'].values])
ey  = nd['Daily_Revenue'].values - Xc @ np.linalg.lstsq(Xc, nd['Daily_Revenue'].values, rcond=None)[0]
ev  = nd['Lag1'].values           - Xc @ np.linalg.lstsq(Xc, nd['Lag1'].values, rcond=None)[0]
r_partial, p_partial = stats.pearsonr(ey, ev)

ax.text(0.04, 0.93,
        f'Raw r = {r_raw:.3f}  |  Partial r (|weekend) = {r_partial:.3f}, p < 0.001  (N = {len(nd)})',
        transform=ax.transAxes, fontsize=9, color=SLATE,
        bbox=dict(boxstyle='round,pad=0.35', facecolor='white', alpha=0.85, edgecolor=MID))
ax.set_xlabel('Visual_Lag1 (CLIP Score, previous day)', fontsize=10)
ax.set_ylabel('Daily Revenue (€)', fontsize=10)
ax.set_title('Visual_Lag1 vs Daily Revenue (Non-Drop Days)\n'
             f'After controlling for weekend effect, partial r = {r_partial:.3f}',
             fontweight='bold', fontsize=11, pad=10)
ax.legend(fontsize=9, framealpha=0.9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(euro))
plt.tight_layout()
save('fig_ch4_fig42_scatter_lag1_revenue.png')


# ── Figure 4.3: Lag Correlation Chart ─────────────────────────────────
# §4.4 — correlation between revenue and the visual score at lags 0–3.

fig, ax = plt.subplots(figsize=(8, 5.5))
df_lc2 = df79.copy()
df_lc2['Lag1'] = df_lc2['Visual_Score'].shift(1)
df_lc2['Lag2'] = df_lc2['Visual_Score'].shift(2)
df_lc2['Lag3'] = df_lc2['Visual_Score'].shift(3)

results2 = []
for lag_col, label in [('Visual_Score','Lag 0\n(Same Day)'),
                        ('Lag1','Lag 1\n(1 Day Prior)'),
                        ('Lag2','Lag 2\n(2 Days Prior)'),
                        ('Lag3','Lag 3\n(3 Days Prior)')]:
    tmp = df_lc2[['Daily_Revenue', lag_col]].dropna()
    r, p = stats.pearsonr(tmp['Daily_Revenue'], tmp[lag_col])
    results2.append((label, r, p, len(tmp)))

rs2    = [r[1] for r in results2]
ps2    = [r[2] for r in results2]
cols2  = [GREEN if p < 0.05 else GREY for p in ps2]
lbls2  = [r[0] for r in results2]

bars2 = ax.bar(lbls2, rs2, color=cols2, edgecolor='white', lw=1.2, width=0.55, zorder=3)
ax.axhline(0, color='black', lw=0.8)
for bar, r, p in zip(bars2, rs2, ps2):
    yp  = r + (0.012 if r >= 0 else -0.025)
    va2 = 'bottom' if r >= 0 else 'top'
    ax.text(bar.get_x()+bar.get_width()/2, yp,
            f'r = {r:.3f}\n{sig_stars(p)}',
            ha='center', va=va2, fontsize=9, fontweight='bold', color=SLATE)

from matplotlib.patches import Patch as P2
ax.legend(handles=[P2(color=GREEN, label='Significant (p < 0.05)'),
                   P2(color=GREY,  label='Not significant')],
          fontsize=9, framealpha=0.9, loc='upper right')
ax.set_ylabel('Pearson r with Daily Revenue', fontsize=10)
ax.set_title('Lag Correlation: Visual Score vs Daily Revenue\n'
             'Carry-over peaks at Lag 1; reverses at Lag 2 and 3',
             fontweight='bold', fontsize=11, pad=10)
ax.set_ylim(-0.35, 0.45)
ax.text(0.01, 0.01, '*** p<0.001  ** p<0.01  * p<0.05  ns = not significant',
        transform=ax.transAxes, fontsize=7.5, color='#666666')
plt.tight_layout()
save('fig_ch4_fig43_lag_correlation_chart.png')


# ── Figure 4.4: Three-Panel Time Series ───────────────────────────────
# §4.5 — revenue, spend, and visual score together in one view.

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
fig.suptitle('Revenue, Ad Spend, and Visual Score Over Time',
             fontweight='bold', fontsize=12, y=1.01)

# Panel 1: Revenue
ax = axes[0]
ax.plot(df79['Date'], df79['Daily_Revenue'], color=SLATE, lw=1.6, zorder=3)
ax.fill_between(df79['Date'], 0, df79['Daily_Revenue'], alpha=0.10, color=SLATE)
for i, dd in enumerate(DROP_DATES, 1):
    ax.axvline(dd, color=RED, lw=1.4, linestyle=':', alpha=0.7)
    rv = df79.loc[df79['Date']==dd, 'Daily_Revenue'].values[0]
    ax.scatter([dd], [rv], color=RED, s=65, zorder=5)
    ax.annotate(f'Drop {i}\n€{rv:,.0f}', xy=(dd, rv),
                xytext=(10, -18), textcoords='offset points',
                fontsize=7.5, color=RED,
                arrowprops=dict(arrowstyle='->', color=RED, lw=0.8))
ax.set_ylabel('Daily Revenue (€)', fontsize=9.5)
ax.yaxis.set_major_formatter(plt.FuncFormatter(euro))
ax.set_title('Revenue', fontsize=10, loc='left', pad=3)

# Panel 2: Ad Spend
mean_nd = df79[df79['Is_Drop'] == 0]['Ad_Spend'].mean()
ax = axes[1]
ax.bar(df79['Date'], df79['Ad_Spend'], color=TEAL, alpha=0.70, width=0.9, zorder=3)
ax.axhline(mean_nd, color=GOLD, lw=1.4, linestyle='--',
           label=f'Non-drop mean (€{mean_nd:.0f})')
for dd in DROP_DATES:
    ax.axvline(dd, color=RED, lw=1.4, linestyle=':', alpha=0.7)
ax.set_ylabel('Ad Spend (€)', fontsize=9.5)
ax.yaxis.set_major_formatter(plt.FuncFormatter(euro))
ax.legend(fontsize=8.5, loc='upper left', framealpha=0.85)
ax.set_title('Advertising Spend', fontsize=10, loc='left', pad=3)

# Panel 3: Visual Score
ax = axes[2]
posting_79 = df79[df79['is_new']]
ax.plot(df79['Date'], df79['Visual_Score'],
        color=GREY, lw=1.0, linestyle='--', zorder=2, label='Forward-filled')
ax.scatter(posting_79['Date'], posting_79['Visual_Score'],
           color=TEAL, s=40, zorder=4, label='New content posted')
ax.fill_between(df79['Date'], df79['Visual_Score'], alpha=0.08, color=TEAL)
ax.axvspan(pd.Timestamp('2025-11-20'), pd.Timestamp('2025-11-24'),
           alpha=0.14, color=RED, label='Low-score episode (20–24 Nov)')
ax.axhline(df79['Visual_Score'].mean(), color=GOLD, lw=1.2, linestyle='--',
           alpha=0.7, label=f'Mean ({df79["Visual_Score"].mean():.3f})')
for i, dd in enumerate(DROP_DATES, 1):
    ax.axvline(dd, color=RED, lw=1.4, linestyle=':', alpha=0.7)
    ax.text(dd, 1.05, f'Drop {i}', ha='center', fontsize=7.5, color=RED,
            transform=ax.get_xaxis_transform())
ax.set_ylabel('CLIP Visual Score', fontsize=9.5)
ax.set_ylim(-0.05, 1.12)
ax.legend(fontsize=8, ncol=2, framealpha=0.85)
ax.set_title('Visual Score (solid = posting day, dashed = forward-filled)',
             fontsize=10, loc='left', pad=3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
save('fig_ch4_fig44_three_panel_timeseries.png')


# ── Figure 4.5: Drop Event Comparison ─────────────────────────────────
# §4.6 — three-panel side-by-side drop analysis.

drop_rows = [df79[df79['Date']==dd].iloc[0] for dd in DROP_DATES]
drop_labels = ['Drop 1\n24 Oct', 'Drop 2\n28 Nov', 'Drop 3\n12 Dec']
x = np.arange(3)

fig, axes = plt.subplots(1, 3, figsize=(13, 5))
fig.suptitle('Drop Event Comparison',
             fontweight='bold', fontsize=12, y=1.01)
colors_d = [TEAL, SLATE, GOLD]

# Left: drop-day revenue vs 7-day pre-drop mean
ax = axes[0]
pre_means = []
for dd in DROP_DATES:
    pre = df79[(df79['Date'] < dd) &
               (df79['Date'] >= dd - pd.Timedelta(days=7)) &
               (df79['Is_Drop']==0)]['Daily_Revenue']
    pre_means.append(pre.mean())

drop_revs = [r['Daily_Revenue'] for r in drop_rows]
width = 0.35
ax.bar(x - width/2, pre_means, width, color=GREY, label='7-day pre-drop mean', alpha=0.8)
ax.bar(x + width/2, drop_revs, width, color=colors_d, label='Drop-day revenue', alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels(drop_labels)
ax.set_ylabel('Revenue (€)', fontsize=9.5)
ax.set_title('Drop-Day Revenue\nvs Pre-Drop Mean', fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(euro))
ax.legend(fontsize=8.5)
for xi, rv, pm in zip(x, drop_revs, pre_means):
    lift = (rv/pm - 1)*100
    ax.text(xi + width/2, rv + 100, f'+{lift:.0f}%', ha='center', fontsize=8,
            color=SLATE, fontweight='bold')

# Centre: drop-day ad spend vs non-drop mean
ax = axes[1]
drop_spends = [r['Ad_Spend'] for r in drop_rows]
mean_nd_s   = df79[df79['Is_Drop']==0]['Ad_Spend'].mean()
ax.bar(x, drop_spends, color=colors_d, alpha=0.9, edgecolor='white', lw=1)
ax.axhline(mean_nd_s, color=GREY, lw=1.6, linestyle='--',
           label=f'Non-drop mean €{mean_nd_s:.0f}')
ax.set_xticks(x); ax.set_xticklabels(drop_labels)
ax.set_ylabel('Ad Spend (€)', fontsize=9.5)
ax.set_title('Drop-Day Ad Spend\nvs Non-Drop Mean', fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(euro))
ax.legend(fontsize=8.5)
for xi, sp in zip(x, drop_spends):
    ax.text(xi, sp + 15, f'€{sp:,.0f}', ha='center', fontsize=8.5,
            color=SLATE, fontweight='bold')

# Right: Visual Score 7 days before each drop
ax = axes[2]
for i, (dd, col) in enumerate(zip(DROP_DATES, colors_d)):
    window = df79[(df79['Date'] >= dd - pd.Timedelta(days=6)) &
                  (df79['Date'] <= dd)]
    offsets = (window['Date'] - dd).dt.days
    ax.plot(offsets, window['Visual_Score'], color=col, lw=2,
            marker='o', ms=5, label=drop_labels[i].replace('\n', ' '))
ax.axvline(0, color='black', lw=1, linestyle=':', alpha=0.5)
ax.text(0.5, -0.09, 'Drop Day', ha='center', fontsize=8.5,
        transform=ax.get_xaxis_transform())
ax.set_xlabel('Days before drop', fontsize=9.5)
ax.set_ylabel('CLIP Visual Score', fontsize=9.5)
ax.set_title('Visual Score Trajectory\n7 Days Pre-Drop', fontsize=10)
ax.legend(fontsize=8.5, framealpha=0.9)
ax.set_ylim(0, 1.1)

plt.tight_layout()
save('fig_ch4_fig45_drop_event_comparison.png')

## Chapter 5 figures

In [ ]:
# ══════════════════════════════════════════════════════════════════════
print('\n── Chapter 5 ──')

# ── Figure 5.1: Residual Diagnostics ──────────────────────────────────
# §5.4 — four-panel diagnostic for Model 3.

e3    = m3['e']
yhat3 = m3['yhat']
drop_mask = df['Is_Drop'].values == 1

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Model 3: Residual Diagnostics',
             fontweight='bold', fontsize=12, y=1.01)

# Panel 1: Residuals vs Fitted
ax = axes[0,0]
ax.scatter(yhat3[~drop_mask], e3[~drop_mask],
           color=TEAL, alpha=0.6, s=40, zorder=3, label='Non-drop days')
ax.scatter(yhat3[drop_mask],  e3[drop_mask],
           color=RED, s=90, zorder=5, marker='D', label='Drop days')
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('Fitted Values (€)', fontsize=9.5)
ax.set_ylabel('Residuals (€)', fontsize=9.5)
ax.set_title('Residuals vs Fitted', fontsize=10)
ax.xaxis.set_major_formatter(plt.FuncFormatter(euro))
ax.yaxis.set_major_formatter(plt.FuncFormatter(euro))
ax.legend(fontsize=8.5)
ax.text(0.04, 0.95, f'DW = {m3["dw"]:.3f}',
        transform=ax.transAxes, fontsize=9, color=SLATE,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85, edgecolor=MID))

# Panel 2: Q-Q plot
ax = axes[0,1]
(osm, osr), (slope, intercept, r) = stats.probplot(e3, fit=True)
ax.plot(osm, osr, color=TEAL, alpha=0.75, linestyle='none', marker='o', ms=5)
ax.plot(osm, slope*np.array(osm)+intercept, color=RED, lw=1.8)
ax.set_xlabel('Theoretical Quantiles', fontsize=9.5)
ax.set_ylabel('Sample Quantiles', fontsize=9.5)
ax.set_title('Q–Q Plot of Standardised Residuals', fontsize=10)
sw, swp = stats.shapiro(e3)
ax.text(0.04, 0.95, f'Shapiro–Wilk W = {sw:.4f}\np = {swp:.4f}',
        transform=ax.transAxes, fontsize=9, color=SLATE,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85, edgecolor=MID))

# Panel 3: Histogram of residuals
ax = axes[1,0]
ax.hist(e3, bins=16, color=TEAL, edgecolor='white', alpha=0.8, density=True)
x_norm = np.linspace(e3.min(), e3.max(), 200)
ax.plot(x_norm, stats.norm.pdf(x_norm, e3.mean(), e3.std()),
        color=RED, lw=1.8, label='Normal curve')
ax.set_xlabel('Residual (€)', fontsize=9.5)
ax.set_ylabel('Density', fontsize=9.5)
ax.set_title('Residual Distribution', fontsize=10)
ax.legend(fontsize=8.5)
ax.xaxis.set_major_formatter(plt.FuncFormatter(euro))

# Panel 4: Residuals over time
ax = axes[1,1]
ax.plot(df['Date'], e3, color=SLATE, lw=1.2, zorder=2)
ax.scatter(df.loc[~drop_mask, 'Date'], e3[~drop_mask],
           color=TEAL, s=28, alpha=0.65, zorder=3)
ax.scatter(df.loc[drop_mask,  'Date'], e3[drop_mask],
           color=RED, s=80, zorder=5, marker='D', label='Drop days')
ax.axhline(0, color='black', lw=0.8)
for dd in DROP_DATES:
    ax.axvline(dd, color=RED, lw=1.2, linestyle=':', alpha=0.6)
ax.set_ylabel('Residual (€)', fontsize=9.5)
ax.set_title('Residuals Over Time', fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(euro))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.legend(fontsize=8.5)

fig.text(0.5, -0.01,
         f'Breusch–Pagan LM = {bp_stat:.3f}, p = {bp_p:.4f}  |  '
         f'Heteroskedasticity driven by three drop-day outliers.',
         ha='center', fontsize=8.5, color='#555555')
plt.tight_layout()
save('fig_ch5_fig51_residual_diagnostics.png')


# ── Figure 5.2: Drop Event Actual vs Fitted + Decomposition ───────────
# §5.7 — ±7 day windows and stacked bar decomposition.

fig = plt.figure(figsize=(14, 6))
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)
fig.suptitle('Drop Events: Actual vs Fitted Revenue and Decomposition',
             fontweight='bold', fontsize=11, y=1.02)

# Left: time-window plots
ax1 = fig.add_subplot(gs[0])
colors_52 = [TEAL, SLATE, GOLD]
drop_label_52 = ['Drop 1 (24 Oct)', 'Drop 2 (28 Nov)', 'Drop 3 (12 Dec)']
for i, dd in enumerate(DROP_DATES):
    window = df[(df['Date'] >= dd-pd.Timedelta(days=7)) &
                (df['Date'] <= dd+pd.Timedelta(days=6))].copy()
    window['offset'] = (window['Date'] - dd).dt.days
    col = colors_52[i]
    ax1.plot(window['offset'], window['Daily_Revenue'],
             color=col, lw=1.8, alpha=0.9, label=f'{drop_label_52[i]} actual')
    fitted_w = [m3['yhat'][df[df['Date']==r['Date']].index[0]]
                for _, r in window.iterrows()]
    ax1.plot(window['offset'], fitted_w,
             color=col, lw=1.2, linestyle='--', alpha=0.65)

ax1.axvline(0, color='black', lw=1.2, linestyle=':', alpha=0.6)
ax1.text(0.5, -0.07, 'Drop Day', ha='center', fontsize=8.5,
         color='black', transform=ax1.get_xaxis_transform())
ax1.set_xlabel('Days relative to drop date', fontsize=9.5)
ax1.set_ylabel('Daily Revenue (€)', fontsize=9.5)
ax1.set_title('Actual (solid) vs Fitted (dashed)\n±7 days around each drop',
              fontsize=10, pad=6)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(euro))
ax1.legend(fontsize=8, framealpha=0.9)

# Right: stacked bar decomposition on drop days
ax2 = fig.add_subplot(gs[1])
x52 = np.arange(3)
width52 = 0.5
comp_names52 = ['Intercept', 'Ad_Spend', 'Is_Weekend', 'Is_Drop', 'Visual_Lag1']
comp_cols52  = {
    'Intercept':   GOLD,
    'Ad_Spend':    SLATE,
    'Is_Weekend':  GREY,
    'Is_Drop':     RED,
    'Visual_Lag1': GREEN
}
b52 = m3['b']
bottoms52 = np.zeros(3)
for ci, cname in enumerate(comp_names52):
    vals52 = []
    for dd in DROP_DATES:
        row52 = df[df['Date']==dd].iloc[0]
        if cname == 'Intercept':
            vals52.append(b52[0])
        elif cname == 'Ad_Spend':
            vals52.append(b52[1]*row52['Ad_Spend'])
        elif cname == 'Is_Weekend':
            vals52.append(b52[2]*row52['Is_Weekend'])
        elif cname == 'Is_Drop':
            vals52.append(b52[3]*row52['Is_Drop'])
        elif cname == 'Visual_Lag1':
            vals52.append(b52[4]*row52['Lag1'])
    vals52 = np.array(vals52)
    bars52 = ax2.bar(x52, vals52, width52, bottom=bottoms52,
                     color=comp_cols52[cname], label=cname,
                     edgecolor='white', lw=0.8)
    for bar52, v52 in zip(bars52, vals52):
        if v52 > 400:
            ax2.text(bar52.get_x()+bar52.get_width()/2,
                     bar52.get_y()+v52/2,
                     f'€{v52:,.0f}', ha='center', va='center',
                     fontsize=7.5, color='white', fontweight='bold')
    bottoms52 += vals52

actuals52 = [df[df['Date']==dd]['Daily_Revenue'].values[0] for dd in DROP_DATES]
ax2.scatter(x52, actuals52, color='black', s=65, zorder=6,
            marker='D', label='Actual revenue')
ax2.set_xticks(x52)
ax2.set_xticklabels(['Drop 1\n24 Oct', 'Drop 2\n28 Nov', 'Drop 3\n12 Dec'], fontsize=9)
ax2.set_ylabel('Revenue (€)', fontsize=9.5)
ax2.set_title('Fitted Revenue Decomposition\nby Component', fontsize=10, pad=6)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(euro))
ax2.legend(fontsize=8, framealpha=0.9, loc='upper left')

plt.tight_layout()
save('fig_ch5_fig52_drop_decomposition.png')

## Thesis tables (CSV)

Every table in the thesis is written to `results/tables/` with the values formatted exactly as printed in the thesis.


In [ ]:
def write_table(name, rows, columns):
    t = pd.DataFrame(rows, columns=columns)
    t.to_csv(TABLE_DIR / f'{name}.csv', index=False)
    print(f'  ✓  results/tables/{name}.csv')
    display(t)

eur  = lambda v, d=2: ('−' if v < 0 else '') + f'€{abs(v):,.{d}f}'
num  = lambda v, d=3: ('−' if v < 0 else '') + f'{abs(v):,.{d}f}'
from decimal import Decimal, ROUND_HALF_UP
score3 = lambda v: str(Decimal(str(v)).quantize(Decimal('0.001'), rounding=ROUND_HALF_UP))   # 0.7535 -> 0.754

# Table 3.1: drop events
rows = []
for i, d in enumerate(DROP_DATES, 1):
    r = df79.loc[df79['Date'] == d].iloc[0]
    pre = df79[(df79['Date'] < d) & (df79['Date'] >= d - pd.Timedelta(days=7)) & (df79['Is_Drop'] == 0)]['Daily_Revenue'].mean()
    rows.append([str(i), d.strftime('%d %b %Y'), r['DayOfWeek'], f"{r['Ad_Spend']:,.2f}", f"{r['Daily_Revenue']:,.2f}",
                 score3(r['Visual_Score']), f'{pre:,.2f}', f"+{(r['Daily_Revenue'] / pre - 1) * 100:.0f}%"])
write_table('table31_drop_events', rows,
            ['Drop', 'Date', 'Day', 'Ad spend (€)', 'Revenue (€)', 'Visual score', 'Pre-drop 7-day mean (€)', 'Lift'])

# Table 4.1: correlation matrix (r above the diagonal, p below)
vars41 = ['Daily_Revenue', 'Ad_Spend', 'Visual_Score', 'Is_Weekend', 'Is_Drop']
rows = []
for i, a in enumerate(vars41):
    row = [a]
    for j, b in enumerate(vars41):
        if i == j:
            row.append('1')
        else:
            r, p = stats.pearsonr(df79[a], df79[b])
            row.append(num(r) + sig_stars(p).replace('ns', '') if i < j else ('< 0.001' if p < 0.001 else f'{p:.3f}'))
    rows.append(row)
write_table('table41_correlation_matrix', rows, [''] + vars41)

# Table 5.1: model comparison
rows = []
for lbl, m, preds in [('M1', m1, 'Ad_Spend, Is_Weekend, Is_Drop'),
                      ('M2', m2, 'M1 + Visual_Score'),
                      ('M3', m3, 'M1 + Visual_Lag1'),
                      ('M4', m4, 'Adstock, Is_Weekend, Is_Drop, Visual_Lag1')]:
    dr = m['r2'] - m1['r2']
    rows.append([lbl, preds, f"{m['r2']:.3f}", f"{m['r2a']:.3f}", f"{m['aic']:.1f}", f"{m['bic']:.1f}",
                 f"{m['rmse']:.2f}", f"{m['dw']:.3f}", '–' if m is m1 else f'+{dr:.3f}'])
write_table('table51_model_comparison', rows,
            ['Model', 'Predictors', 'R²', 'Adj. R²', 'AIC', 'BIC', 'RMSE (€)', 'DW', 'ΔR² vs M1'])

# Table 5.2: Model 3 coefficients
rows = []
for k, nm in enumerate(['Intercept', 'Ad_Spend', 'Is_Weekend', 'Is_Drop', 'Visual_Lag1']):
    b, se, seh, th, ph = m3['b'][k], m3['se'][k], m3['se_h'][k], m3['t_h'][k], m3['pv_h'][k]
    lo, hi = b - t_crit * seh, b + t_crit * seh
    rows.append([nm, num(b, 2), f'{se:,.2f}', f'{seh:,.2f}', f'{th:.2f}', fmt_p(ph) + sig_stars(ph).replace('ns', ''),
                 f'[{num(lo, 2)}, {num(hi, 2)}]'])
write_table('table52_coefficients', rows,
            ['Predictor', 'β', 'OLS SE', 'HC3 SE', 't (HC3)', 'p (HC3)', '95% CI (HC3)'])

# Table 5.3: visual ROI conversion
bv, bs = m3['b'][4], m3['b'][1]
gap = df['Lag1'].quantile(.75) - df['Lag1'].quantile(.25)
rows = []
for lbl, d in [('+0.01', 0.01), ('+0.05', 0.05), ('+0.10', 0.10), (f'+{gap:.3f} (P25 → P75)', gap), ('+0.30', 0.30)]:
    rows.append([lbl, eur(bv * d), eur(bv * d * 365, 0), eur(bv * d / bs)])
write_table('table53_visual_roi', rows, ['Score gain', 'Daily revenue gain', 'Annualised (×365)', 'Ad spend equivalent'])

# Table 5.4: coefficient stability, Model 3 vs Model 4
b3s, b4s = m3['b'][4], m4['b'][4]
ci = lambda m: f"[{eur(m['b'][4] - t_crit * m['se_h'][4])}, {eur(m['b'][4] + t_crit * m['se_h'][4])}]"
d = b4s - b3s
rows = [['β (Visual_Lag1)', eur(b3s), eur(b4s), f"{'−' if d < 0 else '+'}€{abs(d):,.2f} ({'−' if d < 0 else '+'}{abs(d / b3s * 100):.1f}%)"],
        ['OLS SE', eur(m3['se'][4]), eur(m4['se'][4]), ''],
        ['HC3 SE', eur(m3['se_h'][4]), eur(m4['se_h'][4]), ''],
        ['95% CI (HC3)', ci(m3), ci(m4), ''],
        ['VIF', f"{VIF_M3['Visual_Lag1']:.2f}", f'{VIF_M4_vis:.2f}', '']]
write_table('table54_coefficient_stability', rows, ['', 'Model 3', 'Model 4 (adstock)', 'M4 − M3'])
toc('Figures and tables')


## Stage 4 — Key-number check against the thesis

Recomputes every headline number quoted in the thesis and compares it with the value as written, at the precision used in the text. If `master_dataset.csv` is unchanged, every line shows ✓. If anything changed (for example after re-scoring with CLIP), the table lists exactly which numbers differ from the thesis. All values are also saved to `results/key_numbers.json`.

In [ ]:
b3 = m3['b']; t_crit = stats.t.ppf(0.975, n - 5)
p25, p75 = df['Lag1'].quantile(0.25), df['Lag1'].quantile(0.75)
lift = np.maximum((df['Lag1'] - p25) * b3[4], 0).sum()
H3 = X3 @ np.linalg.inv(X3.T @ X3) @ X3.T
loo_rmse = np.sqrt(((m3['e'] / (1 - np.diag(H3))) ** 2).mean())
nd = df[df['Is_Drop'] == 0]
Xn = np.column_stack([np.ones(len(nd)), nd['Ad_Spend'], nd['Is_Weekend'], nd['Lag1']])
b_nd = np.linalg.lstsq(Xn, nd['Daily_Revenue'].values, rcond=None)[0]
lag_r = [stats.pearsonr(*df79[['Daily_Revenue']].assign(v=df79['Visual_Score'].shift(k)).dropna().values.T)[0] for k in range(4)]
comp_mean = {'vis': b3[4] * df['Lag1'].mean(), 'spend': b3[1] * df['Ad_Spend'].mean()}
pred_mean = b3[0] + b3[1]*df['Ad_Spend'].mean() + b3[2]*df['Is_Weekend'].mean() + b3[3]*df['Is_Drop'].mean() + comp_mean['vis']
nd79 = df79[df79['Is_Drop'] == 0]['Daily_Revenue']

# name: (current value, value as written in the thesis, decimals compared)
KEY = {
 'R² M1':                     (m1['r2'], 0.924, 3),
 'R² M2':                     (m2['r2'], 0.945, 3),
 'R² M3':                     (m3['r2'], 0.984, 3),
 'R² M4':                     (m4['r2'], 0.978, 3),
 'Adj. R² M3':                (m3['r2a'], 0.983, 3),
 'AIC M1':                    (m1['aic'], 955.6, 1),
 'AIC M2':                    (m2['aic'], 932.2, 1),
 'AIC M3':                    (m3['aic'], 838.0, 1),
 'AIC M4':                    (m4['aic'], 860.9, 1),
 'BIC M3':                    (m3['bic'], 849.7, 1),
 'RMSE M1 (€)':               (m1['rmse'], 434.70, 2),
 'RMSE M2 (€)':               (m2['rmse'], 369.29, 2),
 'RMSE M3 (€)':               (m3['rmse'], 201.83, 2),
 'RMSE M4 (€)':               (m4['rmse'], 233.79, 2),
 'Durbin–Watson M1':          (m1['dw'], 1.204, 3),
 'Durbin–Watson M2':          (m2['dw'], 2.306, 3),
 'Durbin–Watson M3':          (m3['dw'], 2.182, 3),
 'Partial R² (ΔR²)':          (m3['r2'] - m1['r2'], 0.0597, 4),
 'F-test M1 vs M3':           (F_13, 265.65, 2),
 'β Intercept (€)':           (b3[0], 289.52, 2),
 'β Ad_Spend (€ per €)':      (b3[1], 2.709, 3),
 'β Is_Weekend (€)':          (b3[2], 365.69, 2),
 'β Is_Drop (€)':             (b3[3], 5287.53, 2),
 'β Visual_Lag1 (€)':         (b3[4], 1724.85, 2),
 'OLS SE Visual_Lag1':        (m3['se'][4], 105.83, 2),
 'HC3 SE Visual_Lag1':        (m3['se_h'][4], 116.04, 2),
 'HC3 t Visual_Lag1':         (m3['t_h'][4], 14.86, 2),
 'HC3 CI low Visual_Lag1':    (b3[4] - t_crit * m3['se_h'][4], 1493.59, 2),
 'HC3 CI high Visual_Lag1':   (b3[4] + t_crit * m3['se_h'][4], 1956.11, 2),
 'β Visual_Lag1 in M4 (€)':   (m4['b'][4], 1615.07, 2),
 'β Visual_Lag1 non-drop (€)':(b_nd[3], 1658.69, 2),
 'Breusch–Pagan LM':          (bp_stat, 21.241, 3),
 'Shapiro–Wilk W':            (sw_W, 0.987, 3),
 'Shapiro–Wilk p':            (sw_p, 0.644, 3),
 'VIF Ad_Spend':              (VIF_M3['Ad_Spend'], 11.19, 2),
 'VIF Is_Weekend':            (VIF_M3['Is_Weekend'], 3.29, 2),
 'VIF Is_Drop':               (VIF_M3['Is_Drop'], 7.33, 2),
 'VIF Visual_Lag1':           (VIF_M3['Visual_Lag1'], 1.04, 2),
 'Lag 0 r':                   (lag_r[0], 0.222, 3),
 'Lag 1 r':                   (lag_r[1], 0.308, 3),
 'Lag 2 r':                   (lag_r[2], -0.166, 3),
 'Lag 3 r':                   (lag_r[3], -0.153, 3),
 'Partial r non-drop (|wknd)':(r_partial_nd, 0.877, 3),
 'r Ad_Spend ~ Visual_Score': (stats.pearsonr(df79['Ad_Spend'], df79['Visual_Score'])[0], 0.091, 3),
 'r Ad_Spend ~ Is_Drop':      (stats.pearsonr(df79['Ad_Spend'], df79['Is_Drop'])[0], 0.849, 3),
 'Revenue per +0.10 (€)':     (b3[4] * 0.10, 172.49, 2),
 'P25 Visual_Lag1':           (p25, 0.721, 3),
 'P75 Visual_Lag1':           (p75, 0.931, 3),
 'P25→P75 gap (€/day)':       (b3[4] * (p75 - p25), 361.44, 2),
 'Cumulative lift vs P25 (€)':(lift, 17289, 0),
 'Lift share of revenue (%)': (lift / df['Daily_Revenue'].sum() * 100, 7.3, 1),
 'Spend equivalent of +0.10 (€)': (b3[4] * 0.10 / b3[1], 63.67, 2),
 'LOO RMSE M3 (€)':           (loo_rmse, 245.30, 2),
 'Mean daily revenue (€)':    (df79['Daily_Revenue'].mean(), 3021.65, 2),
 'Mean non-drop revenue (€)': (nd79.mean(), 2732.12, 2),
 'Mean Visual_Score':         (df79['Visual_Score'].mean(), 0.780, 3),
 'Median Visual_Score':       (df79['Visual_Score'].median(), 0.874, 3),
}
check = pd.DataFrame([{'Number': k, 'In thesis': d, 'Now': round(v, p), 'OK': '✓' if round(v, p) == round(d, p) else '✗ UPDATE'}
                      for k, (v, d, p) in KEY.items()])
(RESULTS_DIR / 'key_numbers.json').write_text(json.dumps({k: float(v) for k, (v, _, _) in KEY.items()}, indent=1, ensure_ascii=False))
check.to_csv(RESULTS_DIR / 'key_number_check.csv', index=False)
n_bad = (check['OK'] != '✓').sum()
print(f"{len(check) - n_bad} of {len(check)} numbers match the thesis." + ('' if n_bad == 0 else f'  {n_bad} need updating:'))
display(check if n_bad == 0 else check[check['OK'] != '✓'])

## Stage 5 — Computational environment and running times

Records the hardware and software the notebook ran on, and the time taken by each stage, in `results/runtime.json`. These are the figures reported in Section 3.3.5 and Table 2 of the thesis. In Google Colab, *Runtime → View resources* shows the same RAM figure.

In [ ]:
import platform, os, sys, importlib.metadata as _md

def _cpu_model():
    try:
        with open('/proc/cpuinfo') as f:
            for line in f:
                if line.startswith('model name'):
                    return line.split(':', 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or platform.machine()

def _ram_gb():
    try:
        import psutil
        return round(psutil.virtual_memory().total / 1024**3, 1)
    except ImportError:
        try:
            with open('/proc/meminfo') as f:
                kb = int(f.readline().split()[1])
            return round(kb / 1024**2, 1)
        except OSError:
            return None

def _gpu():
    try:
        import torch
        return torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU only)'
    except ImportError:
        return 'torch not installed'

def _ver(pkg):
    try:
        return _md.version(pkg)
    except _md.PackageNotFoundError:
        return None

PACKAGES = ['numpy', 'pandas', 'scipy', 'statsmodels', 'matplotlib', 'opencv-python', 'pillow',
            'torch', 'torchvision', 'clip', 'ftfy', 'regex']
env = {
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'in_colab': 'google.colab' in sys.modules,
    'cpu_model': _cpu_model(),
    'logical_cpus': os.cpu_count(),
    'ram_gb': _ram_gb(),
    'gpu': _gpu(),
    'packages': {k: v for k, v in ((p, _ver(p)) for p in PACKAGES) if v},
    'timings_seconds': TIMINGS,
}
(RESULTS_DIR / 'runtime.json').write_text(json.dumps(env, indent=2))
print(json.dumps(env, indent=2))


## Exploratory charts *(not used in the thesis)*

Working charts kept from the Colab notebook. They are saved to `figures/exploratory/` so they never mix with thesis figures.

In [ ]:
ex = df79.copy()

# Hype cycle: revenue and visual score on twin axes
fig, ax1 = plt.subplots(figsize=(13, 6))
ax1.plot(ex['Date'], ex['Daily_Revenue'], color='black', lw=2, label='Daily revenue')
ax1.set_ylabel('Daily revenue (€)'); ax1.yaxis.set_major_formatter(plt.FuncFormatter(euro))
ax2 = ax1.twinx()
ax2.plot(ex['Date'], ex['Visual_Score'], color=TEAL, ls='--', lw=2, label='Visual score')
ax2.set_ylabel('Visual score (0–1)', color=TEAL)
for d in DROP_DATES:
    ax1.axvline(d, color=RED, ls=':', alpha=0.7)
ax1.set_title('Hype cycle: visual score and revenue around drops', fontweight='bold')
save('hype_cycle.png', EXPL_DIR)

# Revenue heatmap: spend band × visual band
ex['Visual_Bin'] = pd.cut(ex['Visual_Score'], bins=5, labels=['Very low', 'Low', 'Medium', 'High', 'Very high'])
ex['Spend_Bin']  = pd.cut(ex['Ad_Spend'],    bins=5, labels=['Low', 'Medium-low', 'Medium', 'Medium-high', 'High'])
pivot = (ex.pivot_table(values='Daily_Revenue', index='Spend_Bin', columns='Visual_Bin', aggfunc='mean', observed=False, dropna=False)
           .reindex(index=ex['Spend_Bin'].cat.categories, columns=ex['Visual_Bin'].cat.categories))
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(pivot.values, cmap='YlGnBu', aspect='auto')
ax.set_xticks(range(5)); ax.set_xticklabels(pivot.columns); ax.set_yticks(range(5)); ax.set_yticklabels(pivot.index)
for i in range(5):
    for j in range(5):
        v = pivot.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:,.0f}', ha='center', va='center', fontsize=8.5)
plt.colorbar(im, ax=ax, label='Mean daily revenue (€)')
ax.set_xlabel('Visual score band'); ax.set_ylabel('Ad spend band')
ax.set_title('Mean revenue by spend and visual score (empty cells: no days)', fontweight='bold')
save('revenue_heatmap.png', EXPL_DIR)

# Contour: interpolated revenue surface over spend × visual score
from scipy.interpolate import griddata
xi, yi = np.meshgrid(np.linspace(ex['Ad_Spend'].min(), ex['Ad_Spend'].max(), 100),
                     np.linspace(ex['Visual_Score'].min(), ex['Visual_Score'].max(), 100))
zi = griddata((ex['Ad_Spend'], ex['Visual_Score']), ex['Daily_Revenue'], (xi, yi), method='linear')
fig, ax = plt.subplots(figsize=(9, 7))
cs = ax.contourf(xi, yi, zi, levels=15, cmap='viridis')
plt.colorbar(cs, ax=ax, label='Daily revenue (€)')
dr = ex[ex['Is_Drop'] == 1]
ax.scatter(dr['Ad_Spend'], dr['Visual_Score'], color=RED, s=90, edgecolor='white', label='Drop days')
ax.set_xlabel('Daily ad spend (€)'); ax.set_ylabel('Visual score'); ax.legend()
ax.set_title('Revenue surface over spend and visual score (interpolated)', fontweight='bold')
save('revenue_contour.png', EXPL_DIR)

---
**Outputs**

- `figures/` — the thesis figures, named `fig_ch{chapter}_fig{number}_{description}.png`.
- `figures/exploratory/` — working charts not used in the thesis.
- `results/tables/` — every thesis table as CSV.
- `results/` — `key_numbers.json`, `key_number_check.csv` and `runtime.json` (computational environment and stage timings).
